# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Read Dataset 

In [ ]:
df = pd.read_csv("insurance_claims.csv")
df.head()

# Data Exploration 

In [ ]:
df.info()

In [ ]:
df.isna().sum()

# Data Exploration

In [ ]:
df.replace('?', np.nan, inplace = True)
df.head()

In [ ]:
df['collision_type'].value_counts()

In [ ]:
#fill missing values with most common value
df['collision_type'] = df['collision_type'].fillna(df['collision_type'].mode()[0])

In [ ]:
df['collision_type'].value_counts()

In [ ]:
df['property_damage'].value_counts()

In [ ]:
#fill missing values with most common value
df['property_damage'] = df['property_damage'].fillna(df['property_damage'].mode()[0])
df['property_damage'].value_counts()

In [ ]:
df['police_report_available'].value_counts()

In [ ]:
#fill missing values with most common value
df['police_report_available'] = df['police_report_available'].fillna(df['police_report_available'].mode()[0])
df['police_report_available'].value_counts()

In [ ]:
df.drop(columns = ['_c39'], axis = 1, inplace = True)

In [ ]:
df.isna().sum()

In [ ]:
df.columns

In [ ]:
#drop unnecessary columns
drop_list = ['policy_number', 'policy_bind_date', 'policy_state', 'insured_zip', 'incident_location',
            'incident_date', 'incident_state', 'incident_city', 'insured_hobbies',
            'auto_make', 'auto_model', 'auto_year']
df.drop(drop_list, axis = 1, inplace = True)

In [ ]:
df.head()

In [ ]:
#check for correlation
plt.figure(figsize = (18,12))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype = bool))
sns.heatmap(data = corr, mask = mask, annot = True, fmt = '.2g', linewidth = 1)
plt.show()

In [ ]:
df.drop(columns = ['age', 'total_claim_amount'], inplace = True, axis = 1)
df.head()

In [ ]:
df['fraud_reported'].value_counts()

# Splitting Data

In [ ]:
x = df.drop('fraud_reported', axis = 1)
y = df['fraud_reported']

In [ ]:
#encode categorical data
categorical_df = x.select_dtypes(include = ['object'])
categorical_df.head()

In [ ]:
for col in categorical_df.columns:
    print(f"{col}: \n{categorical_df[col].unique()}\n")

In [ ]:
categorical_df = pd.get_dummies(categorical_df)

In [ ]:
categorical_df.head()

In [ ]:
#select numerical data
num_df = x.select_dtypes(include = ['int64', 'float64'])

In [ ]:
num_df.head()

In [ ]:
#combine categorical data + numerical data
x = pd.concat([num_df, categorical_df], axis = 1)

In [ ]:
x.head()

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.25)

In [ ]:
x_train.columns

In [ ]:
num_df = x_train[['months_as_customer', 'policy_deductable', 'policy_annual_premium', 'umbrella_limit', 'capital-gains',
                'capital-loss', 'incident_hour_of_the_day', 'number_of_vehicles_involved',
                 'bodily_injuries', 'witnesses', 'injury_claim', 'property_claim', 'vehicle_claim']]

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_data = scaler.fit_transform(num_df)

In [ ]:
scaled_data

In [ ]:
scaled_num_df = pd.DataFrame(data = scaled_data, columns = num_df.columns, index = x_train.index)
scaled_num_df.head()

In [ ]:
x_train.drop(columns = scaled_num_df.columns, inplace = True)

In [ ]:
x_train = pd.concat([scaled_num_df, x_train], axis = 1)
x_train.head()

# Building Classification Models 

## **1- Decision Tree Classifier**

In [ ]:
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

In [ ]:
DT_Model = DecisionTreeClassifier()
params = {
    'max_depth' : [3,5,7,10],
    'min_samples_leaf' : list(range(2,10)),
    'criterion' : ['gini', 'entropy'],
    'min_samples_split' :list(range(2,10))
}
grid_search = GridSearchCV(DT_Model, params, cv = 5)
grid_search.fit(x_train, y_train)
print(grid_search.best_params_)
print(grid_search.best_score_)

In [ ]:
dt_model = DecisionTreeClassifier(criterion = 'gini', max_depth = 3, min_samples_leaf = 3,
                                min_samples_split = 4)
dt_model.fit(x_train, y_train)
dt_pred = dt_model.predict(x_test)
train_dt_pred = dt_model.predict(x_train)
print("Testing Classification Report DecisionTreeClassifier: ")
print(classification_report(y_test, dt_pred))
print("Trianing Classification Report DecisionTreeClassifier: ")
print(classification_report(y_train, train_dt_pred))

## **2- RandomForestClassifier**

In [ ]:
RF_Model = RandomForestClassifier()
params = {
    'max_depth' : [3,5,7,10],
    'criterion' : ['gini', 'entropy'],
    'min_samples_split' :list(range(2,10))
}
grid_search = GridSearchCV(RF_Model, params)
grid_search.fit(x_train, y_train)
print(grid_search.best_params_)
print(grid_search.best_score_)

In [ ]:
RF_Model = RandomForestClassifier(criterion = 'gini', max_depth = 10, min_samples_split = 6)
RF_Model.fit(x_train, y_train)
rf_pred = RF_Model.predict(x_test)
train_rf_pred = RF_Model.predict(x_train)
print("Testing Classification Report RandomForestClassifier: ")
print(classification_report(y_test, rf_pred))
print("Trianing Classification Report RandomForestClassifier: ")
print(classification_report(y_train, train_rf_pred))

## **3- Gradient Boosting Classifier**

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
GB_Model = GradientBoostingClassifier()
params = {
    'max_depth' : [3,5,7,10],
    'min_samples_split' :list(range(2,10))
}
grid_search = GridSearchCV(GB_Model, params)
grid_search.fit(x_train, y_train)
print(grid_search.best_params_)
print(grid_search.best_score_)

In [ ]:
GB_Model = GradientBoostingClassifier(max_depth = 3, min_samples_split = 7)
GB_Model.fit(x_train, y_train)
gb_pred = GB_Model.predict(x_test)
train_gb_pred = GB_Model.predict(x_train)
print("Testing Classification Report GradiantBoostingClassifier: ")
print(classification_report(y_test, gb_pred))
print("Trianing Classification Report GradiantBoostingClassifier: ")
print(classification_report(y_train, train_gb_pred))

## **4- Xgboost Classifier**

In [ ]:
y_map = {'Y' :1, 'N':0}
y_train = y_train.map(y_map)

In [ ]:
y_test = y_test.map(y_map)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from xgboost import XGBClassifier
xgb_Model = XGBClassifier()
params = {
    'max_depth' : list(range(2,10)),
    'n_estimators' : [10,50,100,130]
}
grid_search = GridSearchCV(xgb_Model, params)
grid_search.fit(x_train, y_train)
print(grid_search.best_params_)
print(grid_search.best_score_)

In [ ]:
xgb_Model = XGBClassifier(max_depth = 2, n_estimators = 10)
xgb_Model.fit(x_train, y_train)
xgb_pred = xgb_Model.predict(x_test)
train_xgb_pred = xgb_Model.predict(x_train)
print("Testing Classification Report XGBClassifier: ")
print(classification_report(y_test, xgb_pred))
print("Trianing Classification Report XGBClassifier: ")
print(classification_report(y_train, train_xgb_pred))

## **5- Vooting Classsifier**

In [ ]:
from sklearn.ensemble import VotingClassifier

classifiers = [('Decision Tree', dt_model), ('Random Forest', RF_Model),
              ('Gradient Boosting', GB_Model), ('XGBOOST', xgb_Model)]

voting_model = VotingClassifier(estimators = classifiers)
voting_model.fit(x_train, y_train)
voting_pred = voting_model.predict(x_test)
train_voting_pred = voting_model.predict(x_train)

print("Testing Classification Report VotingClassifier: ")
print(classification_report(y_test, voting_pred))
print("Trianing Classification Report VotingClassifier: ")
print(classification_report(y_train, train_voting_pred))